# Pace Smoothing in GCP Platform

This notebook shows a streaming telemetry workflow on Google Cloud:
1. Consume GPX-derived telemetry messages already present in Pub/Sub.
2. Parse and timestamp each message for Beam processing.
3. Apply a stateful rolling-window transform to compute speed, pace, and smoothed pace.
4. Write the enriched records to Bigtable for downstream analytics in Running apps.

In [ ]:
!pip install --quiet gpxpy google-cloud-pubsub google-cloud-storage apache-beam[gcp] geopy

import json
import logging
import os
import statistics
from datetime import datetime, timezone

import apache_beam as beam
from apache_beam.coders import FloatCoder, PickleCoder
from apache_beam.io.gcp.bigtableio import WriteToBigTable
from apache_beam.options.pipeline_options import GoogleCloudOptions, PipelineOptions, StandardOptions
from apache_beam.transforms.userstate import ReadModifyWriteStateSpec
from geopy.distance import geodesic
from google.cloud.bigtable import row

logging.getLogger().setLevel(logging.INFO)

# GCP project and resource configuration for Pub/Sub, Dataflow, and Bigtable.
PROJECT_ID = "<your-project-id>"
BUCKET_NAME = "<your-bucket-name>"
FILE_PATH = "<your-gpx-file>"
TOPIC_ID = "<your-topic-id>"
INSTANCE_ID = "<your-instance-id>"
TABLE_ID = "<your-table-id>"
COLUMN_FAMILY = "metrics"
OUTPUT_PREFIX = f"gs://{BUCKET_NAME}/telemetry-output"


os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID

In [ ]:
# The telemetry stream is assumed to already be available in Pub/Sub.
# This notebook focuses on consuming it, applying smoothing, and writing enriched records to Bigtable.

## GCP pipeline architecture

![GCP pipeline architecture](streaming_GPX_GCP_design.png)

This diagram captures the core runtime path for the notebook: telemetry arrives in Pub/Sub, Beam processes it in a streaming Dataflow job, and the enriched metrics are persisted in Bigtable for later analysis.

## How the smoothing transform works

The smoothing step is a stateful Beam transform that processes telemetry messages one by one while retaining a small amount of state across records. For each incoming point, it compares the current message with the previous one to compute:

- distance covered between the two points using geodesic distance
- elapsed time between the two timestamps
- instantaneous speed in km/h
- pace in minutes per kilometer

To reduce jitter from short intervals or noisy GPS samples, the transform keeps a rolling history of the last five pace values and uses their median as the smoothed pace. The output enriches each message with the derived metrics so downstream analytics can consume a cleaner signal without needing a separate post-processing step.

This design is especially useful for streaming telemetry because it avoids buffering the whole run. Instead, it only needs the previous point and a short pace history for the current runner stream.

In [ ]:
class ParseAndTimestamp(beam.DoFn):
    """Decode Pub/Sub messages and attach a timestamp for Beam windowing."""

    def process(self, element):
        try:
            payload = json.loads(element.decode("utf-8"))
            ts = payload.get("timestamp")
            if ts:
                dt = datetime.fromisoformat(ts)
                yield beam.window.TimestampedValue(payload, dt.timestamp())
            else:
                yield beam.window.TimestampedValue(payload, datetime.now(timezone.utc).timestamp())
        except Exception as exc:
            logging.exception("Failed to parse telemetry payload: %s", exc)


class RollingWindowSmoothing(beam.DoFn):
    """Compute speed, pace, cumulative distance, and a rolling-window smoothed pace using per-key Beam state."""

    PREV_POINT_STATE = ReadModifyWriteStateSpec("prev_point", PickleCoder())
    TOTAL_DIST_STATE = ReadModifyWriteStateSpec("total_dist", FloatCoder())
    PACE_HISTORY_STATE = ReadModifyWriteStateSpec("pace_history", PickleCoder())

    def process(
        self,
        element,
        prev_point_state=beam.DoFn.StateParam(PREV_POINT_STATE),
        total_dist_state=beam.DoFn.StateParam(TOTAL_DIST_STATE),
        pace_history_state=beam.DoFn.StateParam(PACE_HISTORY_STATE),
    ):
        curr_point = dict(element)
        prev_point = prev_point_state.read()
        total_dist_km = total_dist_state.read() or 0.0
        pace_history = pace_history_state.read() or []

        speed_kmh = 0.0
        pace_min_km = 0.0

        if prev_point:
            dist_km = geodesic(
                (prev_point["latitude"], prev_point["longitude"]),
                (curr_point["latitude"], curr_point["longitude"]),
            ).kilometers
            total_dist_km += dist_km

            time_diff_hours = (
                datetime.fromisoformat(curr_point["timestamp"]) - datetime.fromisoformat(prev_point["timestamp"])
            ).total_seconds() / 3600.0

            if time_diff_hours > 0 and dist_km > 0:
                speed_kmh = dist_km / time_diff_hours
                pace_min_km = (time_diff_hours * 60.0) / dist_km

        pace_history.append(pace_min_km)
        pace_history = pace_history[-5:]
        pace_history_state.write(pace_history)

        smoothed_pace_min_km = statistics.median(pace_history) if pace_history else pace_min_km

        prev_point_state.write(curr_point)
        total_dist_state.write(total_dist_km)

        curr_point.update(
            {
                "speed_kmh": round(speed_kmh, 2),
                "pace_min_km": round(pace_min_km, 2),
                "smoothed_pace_min_km": round(smoothed_pace_min_km, 2),
                "total_distance_km": round(total_dist_km, 3),
            }
        )

        yield curr_point


class ToBigtableRow(beam.DoFn):
    """Convert each enriched telemetry record into a Bigtable row for storage."""

    def __init__(self, column_family: str):
        self.column_family = column_family

    def process(self, element):
        row_key = f"runner:{element.get('timestamp', 'unknown')}".encode("utf-8")
        bt_row = row.DirectRow(row_key=row_key)
        for key, value in element.items():
            if value is None:
                continue
            # Column family is treated as a logical Bigtable identifier, so it is passed as a string.
            # The column name and cell value are encoded because Bigtable expects bytes for those fields.
            bt_row.set_cell(
                self.column_family,
                key.encode("utf-8"),
                str(value).encode("utf-8"),
                timestamp=datetime.now(timezone.utc)
            )
        yield bt_row

The implementation below shows the streaming transform and the Bigtable sink that persist the enriched telemetry.

In [ ]:

def build_pipeline(project_id: str, bucket_name: str, topic_id: str, instance_id: str, table_id: str) -> None:
    """Build and run the streaming Beam pipeline that reads Pub/Sub input and writes to Bigtable."""
    options = PipelineOptions(flags=["--streaming", "--runner=DataflowRunner"], save_main_session=True)
    options.view_as(StandardOptions).streaming = True

    cloud_options = options.view_as(GoogleCloudOptions)
    cloud_options.project = project_id
    cloud_options.region = "us-central1"
    cloud_options.staging_location = f"gs://{bucket_name}/staging"
    cloud_options.temp_location = f"gs://{bucket_name}/temp"
    cloud_options.job_name = f"pace-smoothing-{datetime.now(timezone.utc).strftime('%Y%m%d%H%M%S')}"

    with beam.Pipeline(options=options) as pipeline:
        processed = (
            pipeline
            | "Read from Pub/Sub" >> beam.io.ReadFromPubSub(topic=f"projects/{project_id}/topics/{topic_id}")
            | "Parse JSON" >> beam.ParDo(ParseAndTimestamp())
            | "Stateful rolling-window smoothing" >> beam.ParDo(RollingWindowSmoothing())
        )

        (
            processed
            | "Create Bigtable rows" >> beam.ParDo(ToBigtableRow(COLUMN_FAMILY))
            | "Write to Bigtable" >> WriteToBigTable(
                project_id=project_id,
                instance_id=instance_id,
                table_id=table_id,
            )
        )


# Example usage:
# telemetry_messages = load_gpx_points(FILE_PATH)
# publish_gpx_telemetry(telemetry_messages, PROJECT_ID, TOPIC_ID)
# build_pipeline(PROJECT_ID, BUCKET_NAME, TOPIC_ID, INSTANCE_ID, TABLE_ID)

## Bigtable design for smoothed pace data

A compact storage layout for the processed stream is:
- Row key: `runner:{timestamp}`
- Column family: `metrics`
- Columns: `speed_kmh`, `pace_min_km`, `smoothed_pace_min_km`, `total_distance_km`, and the original telemetry fields

This layout keeps writes append-friendly and supports efficient time-series lookups for downstream analytics and monitoring workflows. In production, the simple `runner:{timestamp}` pattern can become a bottleneck because writes may concentrate in a small number of row ranges and create hot spots. A more scalable design is to include a stable runner identifier and a time bucket in the row key, such as `runner:{runner_id}:{yyyyMMddHH}` or a hashed variant, so writes are distributed more evenly across Bigtable tablets.